# 第 12 周 · 周四：综合模拟分析场景——你的"毕业考"

> 难度：⭐⭐⭐⭐⭐ | 定位：综合运用12周全部知识，独立解决一个完整的业务分析问题
> 前置依赖：W1-W12全部内容 | 后续用途：面试case study、真实工作中的第一个独立分析
> 面试地位：大厂面试中"case study"环节直接考察的就是你今天要练的能力

---

## 📘 区块1：核心概念与第一性原理

### 1.1 为什么需要"综合模拟"？

过去11周，你在"分模块"学习——这周学指标拆解，那周学AB实验。但真实世界不按模块出题。CEO不会说"请用窗口函数帮我算一个留存率"——CEO说"DAU涨了但GMV没涨，帮我搞清楚怎么回事"。

综合模拟的本质是**让你在安全环境中体验真实业务问题的解题全过程**——从模糊问题到清晰结论。

### 1.2 从第一性原理推导：解决一个陌生的业务问题需要哪些步骤？

```
模糊的业务问题（"DAU涨了GMV没涨"）
    ↓
步骤1：问题界定——这到底是什么问题？（5分钟）
    不要直接拉数据！先用ISS框架搞清楚：谁的问题？从什么时候开始的？影响了什么？
    ↓
步骤2：分析框架——用什么方法拆解？（10分钟）
    指标拆解(乘法/加法)、AARRR、PDS、同期群……选一个最合适的组合
    ↓
步骤3：假设驱动——最可能的3个原因是什么？（5分钟）
    不是漫无目的地"看数据"，是有目的地"验证假设"
    ↓
步骤4：数据验证——用SQL/Python验证每个假设（核心工作）
    每条SQL都在回答一个具体的假设
    ↓
步骤5：结论+建议——把数据发现翻译成业务决策
    结论优先，给出选项，量化预期效果
```

### 1.3 在这个模拟中你要扮演的角色

你是一家中型电商公司的数据分析师。公司有200万DAU，月GMV约2亿。你的工作直接向CEO汇报。

**TA对你的期待**：
- 不要给ta看数据表——给ta看结论
- 不要用术语——用ta能听懂的语言
- 不要只给一个问题——给选项和建议
- 不要只说"需要进一步分析"——给出你的最佳判断

### 1.4 这个模拟考察的核心能力

| 能力维度 | 具体表现 | 来源 |
|---------|---------|------|
| 问题界定 | 能从模糊描述中精准定义问题 | W1 ISS框架 |
| 指标拆解 | 能把GMV/DAU拆解到可操作的层面 | W1 指标拆解 |
| 框架选择 | 能根据问题特点选择最合适的分析框架 | W1-W3 |
| 假设驱动 | 能按优先级列出最可能的原因并逐一验证 | W10 |
| 数据能力 | 能写出验证假设所需的SQL/Python | W5-W6 |
| 因果判断 | 能区分"相关"和"因果"，避免错误归因 | W7-W9 |
| 表达汇报 | 能用"结论优先"的结构呈现分析结果 | W10/W12D3 |

### 1.5 面试高频考点

**大厂Case Study典型问题**：
- "DAU涨了但GMV没涨，你会怎么分析？"
- "某指标突然下降了15%，你的排查步骤是什么？"
- "给你一个业务场景，设计一个分析方案"
- "CEO让你评估一个业务决策的数据依据"

## 🧩 区块2：分析框架与知识点拆解

### 2.1 核心业务场景

**CEO对你说**：
> "上个月我们的DAU涨了20%，但GMV只涨了3%。运营说是因为新用户质量差，产品说是推荐算法的问题。我不知道该听谁的。帮我搞清楚：钱花在哪了？人为什么买了DAU但没买GMV？我们该怎么做？"

### 2.2 第一步：问题界定（5分钟，用ISS框架）

**I (Identify)——到底发生了什么？**
- DAU涨了20%（从多少到多少？什么时候开始的？）
- GMV只涨了3%（同期对比，正常应该涨多少？）
- 时间范围："上个月"——是指与前一个月环比还是与去年同月同比？

**S (Segment)——问题出在哪一块？**
- GMV = DAU × 付费率 × ARPU
- DAU涨了20%，如果付费率和ARPU不变，GMV应该也涨约20%
- 但GMV只涨了3% → 付费率×ARPU下降了约14%（1.03/1.20 ≈ 0.858）
- 需要确认：是付费率跌了？ARPU跌了？还是都跌了？

**S (Solve)——怎么定位根因？**
- 假设1：新增用户的质量很差（运营说）→ 看新用户vs老用户的付费率和ARPU
- 假设2：推荐算法效果下降（产品说）→ 看推荐位的点击率和转化率趋势
- 假设3：外部因素（竞品促销、季节性等）→ 看大盘数据

### 2.3 第二步：指标拆解树（10分钟）

```
GMV
├── DAU ↑20%（总量涨了，但不是问题所在）
│   ├── 新增用户 ↑↑↑（飙涨！是DAU增长的主因）
│   ├── 留存用户 →（老用户稳定）
│   └── 回流用户 ↓（无变化）
│
├── 付费率 ↓（从4.5%→3.8%，跌了15%）
│   ├── 新用户付费率 ↓↓↓（从3.2%→1.8%，暴跌！）
│   │   ├── 渠道A(信息流)付费率从3%→1.2%
│   │   ├── 渠道B(搜索)付费率从5%→4.8%（略降）
│   │   └── 渠道C(社媒)付费率从3.5%→3.2%（略降）
│   └── 老用户付费率 →（5.2%→5.1%，基本不变）
│
└── ARPU ↓（从85元→81元，降了4.7%）
    ├── 新用户ARPU ↓（从52元→41元）
    └── 老用户ARPU →（91元→90元，基本不变）
```

**关键发现**：DAU涨是因为信息流渠道引入了大量新用户，但这些新用户的付费率和ARPU都远低于老用户——"人来了但不买"。

### 2.4 第三步：假设驱动验证（按优先级）

| 优先级 | 假设 | 验证方法 | 需要的分析 |
|--------|------|---------|-----------|
| 🥇 最高 | 信息流渠道新增用户质量恶化 | 同期群分析：按渠道和注册日期对比新用户的7日/30日付费率 | PDS拆解 + 同期群 + 漏斗 |
| 🥈 高 | 产品推荐算法对这批新用户效果差 | 对比新/老用户在推荐位的CTR和转化率 | 分用户群的推荐位漏斗 |
| 🥉 中 | 竞品在同期做了促销活动 | 看大盘数据和竞品动态 | 同比数据 + 行业报告 |

### 2.5 需要用到的分析框架和方法

| 框架/方法 | 用来做什么 | 来源 |
|----------|-----------|------|
| ISS | 第一步的问题界定和结构拆解 | W3 |
| 指标拆解(乘法) | GMV = DAU × 付费率 × ARPU | W1 |
| PDS | 把DAU涨的来源拆成新增/留存/回流 | W2 |
| 同期群分析 | 按渠道+注册日期看新用户的付费质量 | W2 |
| AARRR漏斗 | 看新增→激活→付费的全链路转化 | W2 |
| 归因模型 | 判断各渠道对GMV的增量贡献 | W3 |
| 漏斗分析 | 定位转化最大的"漏水点" | W2 |
| 异动分析三步法 | 系统化地排查指标异常 | W1 |

## 💻 区块3：动手练习——完整分析流程

### 步骤0：生成模拟数据（真实业务场景）

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)

# 模拟场景：电商公司，DAU涨了20%但GMV只涨了3%
# 生成两个月的用户数据

def generate_monthly_data(n_users_before, n_users_after, ch_quality_shift=True):
    """
    生成模拟用户数据
    ch_quality_shift=True: 新用户在after月的质量变差（模拟信息流渠道质量下降）
    """
    users = []

    for month_label, n_users in [('上月', n_users_before), ('本月', n_users_after)]:
        for i in range(n_users):
            # 用户类型：老用户(60%) vs 渠道新增(40%)
            if np.random.random() < 0.6:
                user_type = '老用户'
                channel = np.random.choice(['自然', '搜索', '社媒'], p=[0.4, 0.35, 0.25])
                # 老用户质量稳定
                pay_prob = 0.052
                arpu_mean = 90
            else:
                user_type = '新用户'
                # 信息流渠道在'本月'质量变差
                channel = np.random.choice(['信息流', '搜索', '社媒'], p=[0.55, 0.25, 0.20])
                if channel == '信息流' and month_label == '本月' and ch_quality_shift:
                    pay_prob = 0.012  # 质量暴跌（标题党素材）
                    arpu_mean = 41
                else:
                    pay_prob = 0.032
                    arpu_mean = 52

            # 是否付费
            paid = np.random.random() < pay_prob
            arpu = np.random.lognormal(np.log(arpu_mean), 0.5) if paid else 0

            users.append({
                'month': month_label,
                'user_type': user_type,
                'channel': channel,
                'paid': paid,
                'amount': round(arpu, 2)
            })

    return pd.DataFrame(users)

df = generate_monthly_data(80000, 96000)  # DAU涨了20%

print("=" * 60)
print("数据生成完成")
print("=" * 60)
print(f"上月活跃用户: 80,000")
print(f"本月活跃用户: 96,000 (+"20%")")
print(f"总样本量: {len(df)}")
print()
print("前10行数据预览：")
print(df.head(10))

### 步骤1：整体指标对比——确认GMV增长远小于DAU增长

In [ ]:
print("=" * 60)
print("步骤1：整体指标对比")
print("=" * 60)

# 按月汇总
monthly = df.groupby('month').agg(
    dau=('user_type', 'count'),
    paying_users=('paid', 'sum'),
    gmv=('amount', 'sum')
).reset_index()

monthly['pay_rate'] = monthly['paying_users'] / monthly['dau'] * 100
monthly['arpu'] = monthly['gmv'] / monthly['dau']

print(monthly.to_string(index=False))
print()

# 计算环比变化
before = monthly[monthly['month'] == '上月'].iloc[0]
after = monthly[monthly['month'] == '本月'].iloc[0]

print("关键变化：")
print(f"  DAU: {before['dau']} → {after['dau']} (+{(after['dau']/before['dau']-1)*100:.1f}%)")
print(f"  GMV: {before['gmv']:.0f} → {after['gmv']:.0f} (+{(after['gmv']/before['gmv']-1)*100:.1f}%)")
print(f"  付费率: {before['pay_rate']:.2f}% → {after['pay_rate']:.2f}%")
print(f"  ARPU: {before['arpu']:.2f} → {after['arpu']:.2f}")
print()

# GMV拆解
gmv_growth = after['gmv'] / before['gmv'] - 1
dau_growth = after['dau'] / before['dau'] - 1
pay_rate_change = after['pay_rate'] / before['pay_rate'] - 1
arpu_change = after['arpu'] / before['arpu'] - 1

print("GMV拆解（乘法）：")
print(f"  GMV变化 = (1+DAU变化) × (1+付费率变化) × (1+ARPU变化) - 1")
print(f"  {gmv_growth*100:.1f}% ≈ (1+{dau_growth*100:.1f}%) × (1+{pay_rate_change*100:.1f}%) × (1+{arpu_change*100:.1f}%) - 1")
print(f"  → 付费率下降是GMV增长乏力的主因！")

### 步骤2：PDS拆解——DAU涨在哪一块？

In [ ]:
print("=" * 60)
print("步骤2：PDS拆解——DAU涨在哪？")
print("=" * 60)

pds = df.groupby(['month', 'user_type']).size().unstack(fill_value=0)
pds_pct = pds.div(pds.sum(axis=1), axis=0) * 100

print("用户构成变化：")
print(pds_pct.round(1).to_string())
print()

print("关键发现：")
print(f"  新用户占比：{pds_pct.loc['上月','新用户']:.1f}% → {pds_pct.loc['本月','新用户']:.1f}%")
print(f"  新用户绝对数：{pds.loc['上月','新用户']} → {pds.loc['本月','新用户']}")
print(f"  → DAU的增长主要由新用户涌入驱动")
print(f"  → 如果新用户质量(付费率)低于老用户，整体付费率就会被稀释")

### 步骤3：同期群分析——新用户vs老用户的付费质量对比

In [ ]:
print("=" * 60)
print("步骤3：同期群分析——新老用户付费质量对比")
print("=" * 60)

cohort = df.groupby(['month', 'user_type']).agg(
    users=('paid', 'count'),
    payers=('paid', 'sum'),
    total_amount=('amount', 'sum')
).reset_index()

cohort['pay_rate'] = (cohort['payers'] / cohort['users'] * 100).round(2)
cohort['avg_arpu'] = (cohort['total_amount'] / cohort['users']).round(2)

print(cohort.to_string(index=False))
print()

print("致命发现：")
before_new = cohort[(cohort['month'] == '上月') & (cohort['user_type'] == '新用户')].iloc[0]
after_new = cohort[(cohort['month'] == '本月') & (cohort['user_type'] == '新用户')].iloc[0]
before_old = cohort[(cohort['month'] == '上月') & (cohort['user_type'] == '老用户')].iloc[0]
after_old = cohort[(cohort['month'] == '本月') & (cohort['user_type'] == '老用户')].iloc[0]

print(f"  新用户付费率：{before_new['pay_rate']}% → {after_new['pay_rate']}%（暴跌！）")
print(f"  新用户ARPU：{before_new['avg_arpu']} → {after_new['avg_arpu']}（暴跌！）")
print(f"  老用户付费率：{before_old['pay_rate']}% → {after_old['pay_rate']}%（稳定 ✓）")
print(f"  老用户ARPU：{before_old['avg_arpu']} → {after_old['avg_arpu']}（稳定 ✓）")
print()
print(f"  → 问题锁定在新用户质量上！老用户一切正常。")

### 步骤4：渠道下钻——哪个渠道的新用户质量出了问题？

In [ ]:
print("=" * 60)
print("步骤4：渠道下钻——定位问题渠道")
print("=" * 60)

new_users = df[df['user_type'] == '新用户']
channel_analysis = new_users.groupby(['month', 'channel']).agg(
    users=('paid', 'count'),
    payers=('paid', 'sum'),
    total_amount=('amount', 'sum')
).reset_index()

channel_analysis['pay_rate'] = (channel_analysis['payers'] / channel_analysis['users'] * 100).round(2)
channel_analysis['avg_arpu'] = (channel_analysis['total_amount'] / channel_analysis['users']).round(2)

print(channel_analysis.to_string(index=False))
print()

# 找出质量下降最严重的渠道
for ch in channel_analysis['channel'].unique():
    before_ch = channel_analysis[(channel_analysis['month'] == '上月') & (channel_analysis['channel'] == ch)]
    after_ch = channel_analysis[(channel_analysis['month'] == '本月') & (channel_analysis['channel'] == ch)]
    if len(before_ch) > 0 and len(after_ch) > 0:
        b = before_ch.iloc[0]
        a = after_ch.iloc[0]
        rate_change = (a['pay_rate'] - b['pay_rate'])
        user_change = (a['users'] - b['users']) / b['users'] * 100
        signal = "🔴🔴🔴" if rate_change < -1 else "🟡" if rate_change < 0 else "🟢"
        print(f"{signal} {ch}:")
        print(f"   用户变化：{b['users']} → {a['users']} (+{user_change:.0f}%)")
        print(f"   付费率变化：{b['pay_rate']}% → {a['pay_rate']}% ({rate_change:+.2f}pp)")
        print(f"   ARPU变化：{b['avg_arpu']} → {a['avg_arpu']}")
        print()

print("→ 信息流渠道：用户暴增但付费率暴跌——典型的'冲量不保质'")
print("→ 搜索和社媒渠道：用户增长合理，质量基本稳定")

### 步骤5：撰写分析报告——结论优先！

In [ ]:
print("=" * 60)
print("📊 分析报告：DAU涨了20%但GMV只涨了3%的根因诊断")
print("=" * 60)
print()
print("🎯 结论：")
print("  信息流渠道换了一批'标题党'素材，DAU虚涨但用户质量断崖下跌。")
print("  产品和运营的直觉都没完全错——运营说的新用户质量差是对的，")
print("  但这些'差用户'主要是信息流一个渠道带来的。")
print()
print("📊 三个核心证据：")
print("  1. 新用户付费率从3.2%→1.8%（-43%），老用户保持稳定(5.2%→5.1%)")
print("  2. 信息流新用户暴增（数量翻倍+），付费率却从3.0%→1.2%")
print("  3. 搜索和社媒渠道的新用户质量基本稳定——问题只出在信息流")
print()
print("💡 行动建议：")
print("  A) 短期（本周）：将信息流素材从'标题党'回滚到'功能卖点'")
print("     预计：DAU可能下降约10%(回到正常水平)，但付费率可恢复至3%+")
print("  B) 长期（下周启动）：对信息流渠道做'素材质量分'监控")
print("     每一项新素材上线前需通过'新用户7日付费率≥2.5%'的检验")
print()
print("⚠ 风险和注意事项：")
print("  - 回滚素材后DAU可能下降——需要提前告知CEO和市场部")
print("  - 信息流用户的'低质'可能也与产品的新手引导有关——建议同步优化")
print("  - 如果素材回滚后付费率仍未恢复→可能是投放人群的问题（需AB实验验证）")

## 📊 区块4：扩展业务场景——其他常见综合模拟题

### 场景1：GMV涨了但利润跌了

**问题**：CEO说"GMV涨了15%，但利润跌了8%。赚了吆喝赔了钱，怎么回事？"

**分析框架**：
```
利润 = 收入 - 成本
     = (GMV × 毛利率) - (固定成本 + 可变成本)
     = (DAU × 付费率 × ARPU × 毛利率) - (固定成本 + 补贴成本 + 获客成本 + ...)
```

**常见根因**：
1. 补贴/折扣率上升 → 毛利率下降 → GMV涨了但到手收入没涨
2. 获客成本飙升 → CAC涨了 → 虽然GMV涨了但增量成本更高
3. 低毛利品类占比上升 → 品类结构变化导致整体毛利率下降
4. 退款率上升 → GMV计入退款前数据 → 实际收入被高估

**分析方法**：
- 拆解GMV的品类构成 → 看各品类毛利率变化
- 同期群分析 → 看不同渠道用户的CAC和LTV
- 补贴/折扣趋势 → 看GMV增长中有多少是"打折买来的"

### 场景2：某A/B实验指标全面"显著"

**问题**：产品经理说"这个实验真厉害，10个指标全显著！"

**你的直觉**：这大概率有问题——10个指标全显著的实验很少见。

**排查清单**：
1. **SRM检查** → 两组人数分布是否符合设计比例？（53:47而50:50 → 分流bug）
2. **Peeking检查** → 实验过程中是否有提前停或反复查看？
3. **多重检验** → 看了10个指标，至少有1个假阳性几乎是确定的（1-0.95^10≈40%）
4. **效应量绝对值** → p值小但效应量可能也很小（大样本下微弱差异也显著）
5. **护栏指标** → 有没有看护栏？（比如转化率涨了但退款率也涨了）

### 场景3：用户增长停滞

**问题**：CTO说"我们最近3个月DAU横盘在200万，怎么突破？"

**分析框架**：
1. **PDS拆解**：DAU = 新增 + 留存 + 回流。哪一块在拖后腿？
2. **如果新增跌了** → 分渠道看新增量 → 是某个渠道出问题了还是整体？
3. **如果留存跌了** → 同期群看哪批用户的留存恶化了 → 新人？以前不这样的老用户？
4. **如果回流跌了** → 看流失用户的特征和流失前的行为 → 能否做召回活动？

### 场景4：评估一次促销活动

**问题**：上月的年中大促，GMV涨了50%——但真的赚钱了吗？

**分析维度**：
1. **增量GMV**：促销期GMV - 如果不促销的"反事实"GMV（用DID或基准模型）
2. **促销成本**：补贴金额 + 额外营销费用 + 额外物流成本
3. **增量利润**：增量GMV × 毛利率 - 促销成本
4. **长期影响**：促销后一个月的用户行为和GMV——有没有"透支"未来消费？
5. **用户质量**：促销期间新增的用户留存如何？——还是"薅一次羊毛就走的"？

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 面对一个模糊的业务问题，能先用ISS框架做问题界定（而不是直接拉数据）
- [ ] 能画出至少两层GMV/DAU的指标拆解树
- [ ] 能用PDS框架拆解DAU变化的来源（新增/留存/回流）
- [ ] 能写出结论优先的分析报告摘要

**答案提示**：
- ISS：Identify(界定)→Segment(拆解定位)→Solve(方案)
- GMV拆解：第一层=DAU×付费率×ARPU，第二层=DAU拆PDS，付费率拆新老用户
- PDS三部分：新增P+留存D+回流S
- 报告摘要模板："建议_____，因为_____(3个数据支撑)，预计带来_____"

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能列出至少5个用于分析复杂业务问题的框架并说明何时用哪个
- [ ] 能按优先级列出3个最可能的假设并设计验证方案
- [ ] 能区分"老用户质量稳定×新用户质量恶化"并量化各自的贡献
- [ ] 能写出5页以内的完整分析报告

**答案提示**：
- 5个框架：ISS(模糊问题)、指标拆解(指标变化)、PDS(DAU波动)、同期群(用户质量)、漏斗(转化环节)
- 假设驱动：优先级=影响程度×置信度（先用简单方法验证影响最大的假设）
- 新老用户分流：按用户注册时间分层对比，用加权法量化新用户质量变化的整体影响
- 5页报告：摘要→证据1→证据2→行动建议→风险与局限

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能在30分钟内完成"问题界定→分析框架→假设→验证方案"的完整设计
- [ ] 能识别团队里其他人在分析同一问题时可能踩的坑
- [ ] 能在面试中面对任意case study问题结构化地回答
- [ ] 能将分析框架迁移到完全不同行业的问题（电商→金融→教育）

**答案提示**：
- 30分钟分配：5min问题界定 → 10min框架选择+指标拆解 → 5min假设驱动 → 10min验证方案设计
- 常见坑：没做问题界定就拉数据、只看总量不看结构、不做新老用户区分、混淆相关和因果
- 面试技巧：先说框架再进入细节，展示系统性思维而非直觉猜测
- 迁移能力：核心框架(ISS/AARRR/同期群/指标拆解)是跨行业的——电商的GMV=金融的交易额

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← W1** | 指标拆解与ISS框架 | 所有分析的起点——先界定问题再拆解 |
| **← W2** | AARRR、同期群、PDS、漏斗 | 本次分析核心使用的框架 |
| **← W3** | 归因模型 | 判断各渠道对GMV的增量贡献 |
| **← W5-W6** | SQL分析能力 | 所有验证假设的查询都依赖SQL |
| **← W7-W9** | AB实验与因果推断 | 判断"换素材"是否真的导致了"用户质量下降" |
| **← 昨日** | 分析表达 | 最终报告怎么写、怎么汇报 |
| **→ 明日** | 后续学习规划 | 今天暴露的薄弱点决定明天规划的方向 |

### 横向连接：完整分析能力的"五边形"

```
          问题界定
            /\
           /  \
    报告撰写    框架选择
         \    /
          \  /
      数据能力 ←→ 因果判断
```

五个维度缺一不可：
- 问题界定弱 → 方向错了再好的分析也没用
- 框架选择弱 → 有方向但不知道该用什么方法
- 数据能力弱 → 想好了验证方法但写不出SQL
- 因果判断弱 → 数据找到了但分不清相关和因果
- 报告撰写弱 → 分析做对了但别人不懂、不信、不动

## 📝 区块7：当日小结

> **面对一个模糊的业务问题（"DAU涨了GMV没涨"），完整分析路径是5步法：①问题界定(ISS，10分钟)——搞清楚到底发生了什么、从什么时候开始、影响了什么 ②指标拆解——GMV=DAU×付费率×ARPU，把宏观问题拆到可操作的层次 ③假设驱动——按优先级列出3个最可能的根因假设，用数据逐一验证 ④下钻定位——PDS看DAU结构→同期群看新老差异→渠道分析定位问题渠道 ⑤结论+建议——一份5页以内的报告：结论→证据→行动建议→风险。核心认知：分析不是"看数据"，而是"验证假设"。不是漫无目的地拉数据，而是像医生一样——先猜可能的病因，再用检验来确认。**

**三条金句带回家**：
1. 收到业务问题时，第一反应不是"我开始拉什么数据"，而是"我用什么框架拆解这个问题"
2. GMV = DAU × 付费率 × ARPU 是电商分析师的"E=mc²"——永远的第一性原理
3. "新用户vs老用户"是最常用的分层维度——大部分"总量看起来正常"的问题，一拆就露馅

### 🔖 课后拓展

1. **练习**：给你自己出一道综合模拟题。选一个你熟悉的产品（微信/淘宝/抖音），造一个"指标异常"的场景，自己走一遍完整的5步分析流程。

2. **思考**：如果你的分析结论是"信息流渠道质量下降"，但市场VP坚持认为"是在拉新，质量不重要，规模起来后会好转"——你会怎么回应？（提示：用LTV/CAC比来算——如果A用户在信息流的获取成本为C，预计LTV是多少？如果是负数——拉新越多赔越多。）

3. **面试准备**：准备好你的"综合分析story"——用S.T.A.R结构，讲述一次你从模糊问题到清晰结论的完整分析经历。控制在3分钟内讲完。